In [0]:
bookings_df = spark.table("dev.spark_db.bookings")
facilities_df = spark.table("dev.spark_db.facilities")
members_df = spark.table("dev.spark_db.members")

df = bookings_df.join(facilities_df , bookings_df.facility_id == facilities_df.facility_id , "inner")\
                      .join(members_df , bookings_df.member_id == members_df.member_id , 'left')\
                      .select(bookings_df.member_id.alias("member_id") , "booking_id" , 
                               "first_name" , "last_name" , "guest_cost" , "member_cost" , "start_time" , "facility_name" , "slots")\
                      .selectExpr("member_id",
                                 "booking_id",
                                 "case when member_id == 0 then 'Guest Name' else concat_ws(' ' , first_name , last_name) end as member_name",
                                 "facility_name",
                                 "start_time",
                                 "case when member_id == 0 then slots * guest_cost else slots * member_cost end as booking_amount")
    
df.display()

In [0]:
from pyspark.sql.functions import month,col,sum,year


df.where(year(df.start_time)== 2022)\
    .withColumn( "month" , month(df.start_time) )\
    .groupBy(col("month")).agg(sum(df.booking_amount).alias("revenue"))\
    .orderBy(col("month").asc())\
    .show()

In [0]:
from pyspark.sql.functions import month,col,sum,year

# groupBy + all agg columns
df.where(year(df.start_time)== 2022)\
    .withColumn( "month" , month(df.start_time) )\
    .rollup(col("month")).agg(sum(df.booking_amount).alias("revenue"))\
    .orderBy(col("month").asc_nulls_last())\
    .show()

In [0]:
from pyspark.sql.functions import expr,col,sum

df.withColumn("revenue_from" , expr("case when member_id == 0 then 'Guest' else 'Member' end"))\
    .groupBy(col("revenue_from") , col("facility_name"))\
    .agg(sum(col("booking_amount")))\
    .orderBy(col("revenue_from").asc_nulls_last() , col("facility_name").asc_nulls_last() )\
    .display()

# group by - 1 dimension

In [0]:
from pyspark.sql.functions import expr,col,sum

df.withColumn("revenue_from" , expr("case when member_id == 0 then 'Guest' else 'Member' end"))\
    .rollup(col("revenue_from") , col("facility_name"))\
    .agg(sum(col("booking_amount")))\
    .orderBy(col("revenue_from").asc_nulls_last() , col("facility_name").asc_nulls_last() )\
    .display()

# 3 dimensions
# rollup 
#        Normal Group by +
#        groupby(revenue_from) -> guest , null(facility_name) , sum(booking_amount)  (independent (no dependencies no facility_name)) + 
#        Total row sum()

# Only looks at the first columns fro the second point to get bot the columns we have to use cube for all possible scenarios

In [0]:
from pyspark.sql.functions import expr,col,sum

df.withColumn("revenue_from" , expr("case when member_id == 0 then 'Guest' else 'Member' end"))\
    .cube(col("revenue_from") , col("facility_name"))\
    .agg(sum(col("booking_amount")))\
    .orderBy(col("revenue_from").asc_nulls_last() , col("facility_name").asc_nulls_last() )\
    .display()

# 4 dimensions
# cube(all dimensions) -  normal group by
#         group by ("revenue_from").sum("booking_amonut")    facility_name = null
#         group by ("facility_name").sum("booking_amount")   revenue_from = null
#         grand total  sum("booking_amount")




In [0]:
from pyspark.sql.functions import expr,col,sum

df.withColumn("revenue_from" , expr("case when member_id == 0 then 'Guest' else 'Member' end"))\
    .groupingSets([ ("revenue_from" , "facility_name") , ("revenue_from" , )  ] , col("revenue_from") , col("facility_name"))\
    .agg(sum(col("booking_amount")))\
    .orderBy(col("revenue_from").asc_nulls_last() , col("facility_name").asc_nulls_last() )\
    .display()

# groupingSets - 2 dimension
  # no grand total
